# PLaTune: Pretrained Latents Tuner

[PLaTune](https://github.com/acids-ircam/platune) is a latent diffusion transformer that adds time-varying musical controls (pitch, octave, onsets, dynamics, instrument, plus optional audio descriptors like loudness / centroid / sharpness) on top of any pretrained neural audio codec — *without* retraining the codec.

Reference: *Adding temporal musical controls on top of pretrained generative models* by Sarah Nabi, Nils Demerlé, Geoffroy Peeters, Frédéric Bevilacqua, and Philippe Esling — [ISMIR 2025](https://ismir2025.ismir.net/).

----

This notebook covers training a PLaTune model on top of [Music2Latent](https://github.com/SonyCSLParis/music2latent) (the codec is downloaded automatically on first use; no separate upload required). It runs on a single Kaggle GPU and supports resume-across-sessions via Kaggle's *Save output as Dataset* pattern.

----

Last updated: 30.04.2026

## Setup runtime

Set up Miniconda with Python 3.12 (PLaTune requires `>=3.12`), then clone PLaTune from GitHub and install its dependencies on the runtime.

In [ ]:
#Install Miniconda

!mkdir /kaggle/temp
%cd /kaggle/temp
!curl -L https://repo.anaconda.com/miniconda/Miniconda3-py312_24.11.1-0-Linux-x86_64.sh -o miniconda.sh
!chmod +x miniconda.sh
!sh miniconda.sh -b -p /kaggle/temp/miniconda

In [ ]:
#Install PLaTune

%cd /kaggle/temp
!git clone https://github.com/skabbit/platune.git
%cd /kaggle/temp/platune
#nn_tilde is only needed for nn~ Max/PD export and often fails to build on Kaggle — drop it
!grep -v '^nn_tilde' requirements.txt > /kaggle/temp/requirements.txt
!/kaggle/temp/miniconda/bin/pip install -r /kaggle/temp/requirements.txt
!/kaggle/temp/miniconda/bin/pip install -e .

In [ ]:
#Compatibility issues: force reinstall of Torch matching PLaTune's pin; pin setuptools<81 so pretty_midi can keep importing pkg_resources
!/kaggle/temp/miniconda/bin/pip install torch==2.7.0 torchvision==0.22.0 torchaudio==2.7.0 --force-reinstall
!/kaggle/temp/miniconda/bin/pip install "setuptools<81" --force-reinstall

In [ ]:
#Patch music2latent for PyTorch >= 2.6: its bundled checkpoint trips the new `weights_only=True` default

!/kaggle/temp/miniconda/bin/python - <<'PY'
import music2latent.inference, pathlib
p = pathlib.Path(music2latent.inference.__file__)
src = p.read_text()
new = src.replace('map_location=self.device)', 'map_location=self.device, weights_only=False)')
p.write_text(new)
print('patched' if new != src else 'no match', p)
PY

## Preprocess dataset

Preprocessing needs to be done once per training (when resuming, deactivate this section). It encodes your audio into Music2Latent latents, runs basic-pitch to derive MIDI, computes the discrete control attributes (pitch / octave / onsets / dynamics / instrument), and writes everything to an LMDB.

Pick the parser that matches your audio collection (`platune/datasets/parsers.py`):
- `simple_parser` — any folder of audio (no instrument label),
- `solo_parser` — single-track / unlabeled set (treated as one `'solo'` instrument; pair with `--config solo`),
- `medley_solos_mono_parser`, `urmp_parser`, `slakh_parser`, `synthetic_parser`, `maestro_parser` — dataset-specific.

Note: a parser that doesn't set `metadata['instrument']` (e.g. `simple_parser`) is incompatible with `--midi_attributes` because the MIDI-attribute step needs an instrument label. Use `solo_parser` if you have one unlabeled source, or write your own parser otherwise.

In [ ]:
#PLaTune dataset preprocessing

!mkdir /kaggle/working/processed

!/kaggle/temp/miniconda/bin/python /kaggle/temp/platune/scripts/prepare_dataset.py \
--input_path /kaggle/input/your-audio-folder \
--output_path /kaggle/working/processed \
--db_size 64 \
--parser_name solo_parser \
--config solo \
--emb_model_path music2latent \
--gpu 0 \
--num_signal 131072 \
--sr 44100 \
--batch_size 32 \
--cut_silences \
--save_waveform \
--use_basic_pitch \
--midi_attributes

## Train PLaTune model

Below section trains a PLaTune model on the LMDB you just built. The shipped `smoke.gin` is sized for music2latent (LATENT_DIM=64, 2-layer transformer) and is what the smoke test verified end-to-end; use `v1.gin` for a larger production-style model.

In order to resume training, transform the output of the first run into a Kaggle Dataset (Output tab → *Save output as Dataset*). Add that dataset to a new run of this notebook, **enable** the `cp` cell below to copy the prior output back into `/kaggle/working/`, **disable** the preprocess section, and add `--ckpt /kaggle/working/runs/your-training-name` to the train command — `train.py` recursively searches the run dir for the latest `*last*.ckpt` and resumes from there.

Keep `--config`, `--name`, and any `--bins_values_file` / `--min_max_file` flags identical to the initial run.

In [ ]:
#Copy files to /kaggle/working folder to continue training from an earlier checkpoint.
#!cp -r /kaggle/input/your-earlier-training-output/* /kaggle/working

In [ ]:
#PLaTune model training. Use --ckpt flag (run dir or .ckpt path) when you continue an earlier training.

!/kaggle/temp/miniconda/bin/python /kaggle/temp/platune/scripts/train.py \
--db_path /kaggle/working/processed \
--name your-training-name \
--config smoke \
--save_path /kaggle/working/runs \
--max_steps 300000 \
--val_every 10000 \
--gpu 0 #\
#--ckpt /kaggle/working/runs/your-training-name